In [1]:
import numpy as np
import networkx as nx
import pickle
from itertools import combinations
import copy

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz,  PauliEvolutionGate, CXGate, SwapGate
from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import (
    HighLevelSynthesis, 
    InverseCancellation
)
from qiskit.transpiler.coupling import CouplingMap

from qiskit.circuit import Parameter

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, CommutingGateRouter, FindCommutingPauliEvolutionsMulti, DecomposePauliZEvolution

from qopt_best_practices.transpilation.qaoa_construction_pass import QAOAConstructionPass
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping


In [3]:
properties = {}
def get_permutation(pass_, dag, time, property_set, count):
    properties["virtual_permutation_layout"] = property_set["virtual_permutation_layout"]

In [4]:
with open('/lustre/scratch127/qpg/jc59/hubo/results_test_N3_W4_extra0_four0.0_six1.0.R2C2.pkl', 'rb') as f:
    data = pickle.load(f)

In [5]:
hamiltonian = data[50]['hamiltonian']
hamiltonian = hamiltonian.sort()

old_hamiltonian = data['old_hamiltonian']
edge_map = data[50]['layout']

In [2]:
extended_swap_strat = ExtendedSwapStrategy.from_heavy_hex(2, 2)
num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map

In [7]:
extended_swap_strat._coupling_map.distance(0,0)

0

In [7]:
edge_map[10] = 14

In [8]:
# The following is useful to remove extra qubits from the circuit. Only meaningful for simulations, but for those we just do a minimal line strategy
current_virtual_qubit_locations = [j for j in edge_map.values()]
used_physical_qubits = set(current_virtual_qubit_locations)
swap_layers: list[tuple[tuple[int, int], ...]] = []
for idx in range(len(extended_swap_strat)):
    swaps_to_perform = [x for x in extended_swap_strat.swap_layer(idx) if any(j in x for j in edge_map.values())]
    swap_layers.append(tuple(swaps_to_perform))
    new_list = copy.copy(current_virtual_qubit_locations)
    for i, j in swaps_to_perform:
        if i in current_virtual_qubit_locations:
            new_list[current_virtual_qubit_locations.index(i)] = j
        if j in current_virtual_qubit_locations:
            new_list[current_virtual_qubit_locations.index(j)] = i
    current_virtual_qubit_locations = new_list
    used_physical_qubits = used_physical_qubits.union(current_virtual_qubit_locations)

len(used_physical_qubits)

20

In [9]:
remapping = {qubit: idx for idx, qubit in enumerate(used_physical_qubits)}

In [10]:
coupling_edges = coupling_map.get_edges()
new_edges = [(remapping[edge[0]], remapping[edge[1]]) for edge in coupling_edges if edge[0] in used_physical_qubits and edge[1] in used_physical_qubits]

In [11]:
remapped_swap_layers = [
    tuple( [(remapping[edge[0]], remapping[edge[1]]) for edge  in swap_layers[l]]) for l in range(len(swap_layers))
]


In [12]:
new_ess = ExtendedSwapStrategy(CouplingMap(new_edges), tuple(remapped_swap_layers))

In [13]:
dual_coupling_map = nx.Graph()
coupling_map_edge = list(new_ess._coupling_map)
physical_qubits = list(new_ess._coupling_map.physical_qubits)
for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)

In [14]:
pm = PassManager(
    [
        HighLevelSynthesis(basis_gates=["PauliEvolution"]), # Not needed if set up circuit as PauliEvolutionGate
        FindCommutingPauliEvolutionsMulti(), 
        CommutingGateRouter(
            new_ess,
            edge_colouring,
            max_layers=50,
            perform_extra_swaps=bool(0)
        ),
        SwapToFinalMapping(),
        DecomposePauliZEvolution(new_ess._coupling_map),
        HighLevelSynthesis(
            basis_gates=["sx", "x", "rz", "rzz", "cx", "id", "swap"], 
        ),
        InverseCancellation(gates_to_cancel=[CXGate(), SwapGate()]),
    ]
)

In [15]:
old_cost_qc = QuantumCircuit(num_physical_qubits)
old_cost_qc.append(PauliEvolutionGate(old_hamiltonian, time=Parameter("c")), [remapping[edge_map[i]] for i in range(len(edge_map))])
old_cost_qc.draw(fold=-1, idle_wires=False)

┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
 q_0: ┤1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [16]:
told_cost_qc = pm.run(old_cost_qc)
# Massively increased the number of failed gates...

10:19:48 - qiskit_qaoa.utils.transpiler_passes - INFO - Max layers needed to apply swap decompose: 43
10:19:48 - qiskit_qaoa.utils.transpiler_passes - INFO - Gates we cannot directly implement: 68
10:19:48 - qiskit_qaoa.utils.transpiler_passes - INFO - [(0, 16, 19), (4, 3, 19, 1, 2, 18), (10, 16, 4, 19), (0, 16, 3), (16, 4, 3, 19), (18, 14, 7, 17), (10, 4, 3, 19), (10, 14), (3, 19, 1), (2, 18, 14, 7, 17), (10, 4, 3), (10, 0, 16, 19), (0, 16, 4, 3, 19), (0, 16, 4, 19), (0, 1, 2), (0, 14, 7), (10, 0, 19), (10, 0, 16, 4, 3, 19), (10, 4), (1, 18, 7, 17), (10, 16, 3, 19), (1, 2, 18, 14, 7), (10, 0, 16, 4, 19), (1, 2, 18, 14), (1, 18, 14, 17), (2, 18, 7), (2, 18, 7, 17), (2, 7, 17), (10, 1, 2), (1, 7, 17), (4, 3, 19, 1), (10, 16, 4, 3, 19), (4, 2, 18), (0, 2), (4, 19, 1, 2, 18), (10, 16, 19), (1, 2, 18, 14, 7, 17), (10, 0, 1, 2), (1, 14, 7, 17), (4, 19, 1, 18), (4, 3, 19, 2, 18), (10, 0, 14, 7), (0, 3, 19), (1, 14, 7), (10, 1), (1, 18, 14, 7, 17), (1, 2, 18, 7, 17), (1, 2, 18, 14, 17), (4, 3

In [17]:
told_cost_qc.draw(fold=-1, idle_wires=False)

global phase: (-30.4375)*c
                                                                                                                                                                                                 ┌────────────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         
  q_0 -> 0 ───────────────■─────────────────────────────────────────────────────────────────────────────────────────────────────────────■─────────────────────────────────────────────────────■──┤ Rz((-1.375)*c) ├───────────────────────────────────────────X────────────────────────────X───────────────────────────────────────────────X────────────────────────────X─────────────────────────────────X────────────────────────────X───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■──────────────────────────■─────────X──────────────────────────────────────■───────────────────────────────────────────────────────■─────────────X────────────────X───────X────────X─────────────────────X─────────────────────────X────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────────────────────────────────■─────────────────X───────X───────X───────X────────────────X───

In [18]:
cost_qc = QuantumCircuit(num_physical_qubits)
cost_qc.append(PauliEvolutionGate(hamiltonian, time=Parameter("c")), range(num_physical_qubits))
tcost_qc = pm.run(cost_qc, callback=get_permutation)
tcost_qc.num_qubits

IndexError: index 30 is out of bounds for axis 0 with size 20

In [ ]:
active_qubits = [int(any([pauli.z[i] for pauli in hamiltonian.paulis])) for i in range(len(hamiltonian.paulis[0].z))]
np.nonzero(active_qubits)

In [ ]:
cost_qc.decompose().draw(fold=-1, idle_wires=False)

In [ ]:
tcost_qc.draw(fold=-1,idle_wires=False)

In [ ]:
def count_gates(qc: QuantumCircuit):
    gate_count = { qubit: 0 for qubit in qc.qubits }
    for gate in qc.data:
        for qubit in gate.qubits:
            gate_count[qubit] += 1
    print(gate_count)
    return gate_count


def remove_idle_wires(qc: QuantumCircuit):
    qc_out = qc.copy()
    gate_count = count_gates(qc)
    for qubit, count in gate_count.items():
        if count == 0:
            print(f'Removing qubit: {qubit}')
            qc_out.qubits.remove(qubit)
    return qc_out

In [ ]:
print(told_cost_qc.num_qubits)
rem = remove_idle_wires(told_cost_qc)

35
{<Qubit register=(35, "q"), index=0>: 51, <Qubit register=(35, "q"), index=1>: 52, <Qubit register=(35, "q"), index=2>: 68, <Qubit register=(35, "q"), index=3>: 71, <Qubit register=(35, "q"), index=4>: 70, <Qubit register=(35, "q"), index=5>: 25, <Qubit register=(35, "q"), index=6>: 49, <Qubit register=(35, "q"), index=7>: 40, <Qubit register=(35, "q"), index=8>: 61, <Qubit register=(35, "q"), index=9>: 18, <Qubit register=(35, "q"), index=10>: 53, <Qubit register=(35, "q"), index=11>: 21, <Qubit register=(35, "q"), index=12>: 59, <Qubit register=(35, "q"), index=13>: 47, <Qubit register=(35, "q"), index=14>: 17, <Qubit register=(35, "q"), index=15>: 62, <Qubit register=(35, "q"), index=16>: 71, <Qubit register=(35, "q"), index=17>: 49, <Qubit register=(35, "q"), index=18>: 43, <Qubit register=(35, "q"), index=19>: 43, <Qubit register=(35, "q"), index=20>: 0, <Qubit register=(35, "q"), index=21>: 0, <Qubit register=(35, "q"), index=22>: 0, <Qubit register=(35, "q"), index=23>: 0, <Q

In [ ]:
rem.qubits

[<Qubit register=(35, "q"), index=0>,
 <Qubit register=(35, "q"), index=1>,
 <Qubit register=(35, "q"), index=2>,
 <Qubit register=(35, "q"), index=3>,
 <Qubit register=(35, "q"), index=4>,
 <Qubit register=(35, "q"), index=5>,
 <Qubit register=(35, "q"), index=6>,
 <Qubit register=(35, "q"), index=7>,
 <Qubit register=(35, "q"), index=8>,
 <Qubit register=(35, "q"), index=9>,
 <Qubit register=(35, "q"), index=10>,
 <Qubit register=(35, "q"), index=11>,
 <Qubit register=(35, "q"), index=12>,
 <Qubit register=(35, "q"), index=13>,
 <Qubit register=(35, "q"), index=14>,
 <Qubit register=(35, "q"), index=15>,
 <Qubit register=(35, "q"), index=16>,
 <Qubit register=(35, "q"), index=17>,
 <Qubit register=(35, "q"), index=18>,
 <Qubit register=(35, "q"), index=19>]

In [ ]:
used_physical_qubits

{4, 7, 8, 9, 10, 12, 13, 14, 15, 21, 22, 24, 26, 27, 28, 29, 30, 32, 33, 34}

In [ ]:
list(new_ess._coupling_map)

[(0, 9),
 (0, 10),
 (1, 11),
 (1, 12),
 (2, 12),
 (2, 13),
 (2, 14),
 (3, 13),
 (3, 15),
 (3, 10),
 (4, 15),
 (4, 16),
 (5, 17),
 (6, 17),
 (6, 18),
 (6, 14),
 (7, 18),
 (7, 19),
 (8, 19),
 (8, 16),
 (9, 0),
 (10, 0),
 (11, 1),
 (12, 1),
 (12, 2),
 (13, 2),
 (14, 2),
 (13, 3),
 (15, 3),
 (10, 3),
 (15, 4),
 (16, 4),
 (17, 5),
 (17, 6),
 (18, 6),
 (14, 6),
 (18, 7),
 (19, 7),
 (19, 8),
 (16, 8)]

In [ ]:

basis_gates=["sx", "x", "rz", "rzz", "cz", "id"]

backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates=basis_gates,
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = len(used_physical_qubits)
config["basis_gates"] = basis_gates
print(config)
config = AerBackendConfiguration.from_dict(config)
print(config.to_dict())
# Why does this have 35 qubits?!?!??!?!?
backend = AerSimulator(configuration=config, coupling_map=new_ess._coupling_map, **backend_options)

{'backend_name': 'aer_simulator', 'backend_version': '0.17.1', 'n_qubits': 20, 'url': 'https://github.com/Qiskit/qiskit-aer', 'simulator': True, 'local': True, 'conditional': True, 'memory': True, 'max_shots': 1000000, 'description': 'A C++ Qasm simulator with noise', 'coupling_map': None, 'basis_gates': ['sx', 'x', 'rz', 'rzz', 'cz', 'id'], 'custom_instructions': ['break_loop', 'continue_loop', 'delay', 'for_loop', 'if_else', 'initialize', 'kraus', 'qerror_loc', 'quantum_channel', 'reset', 'roerror', 'save_amplitudes', 'save_amplitudes_sq', 'save_clifford', 'save_density_matrix', 'save_expval', 'save_expval_var', 'save_matrix_product_state', 'save_probabilities', 'save_probabilities_dict', 'save_stabilizer', 'save_state', 'save_statevector', 'save_statevector_dict', 'save_superop', 'save_unitary', 'set_density_matrix', 'set_matrix_product_state', 'set_stabilizer', 'set_statevector', 'set_superop', 'set_unitary', 'superop', 'switch_case', 'while_loop'], 'gates': []}
{'backend_name': 'a

In [ ]:
backend.configuration().to_dict()

{'backend_name': 'aer_simulator_statevector',
 'backend_version': '0.17.1',
 'n_qubits': 35,
 'basis_gates': ['sx',
  'x',
  'rz',
  'rzz',
  'cz',
  'id',
  'break_loop',
  'continue_loop',
  'delay',
  'for_loop',
  'if_else',
  'initialize',
  'kraus',
  'qerror_loc',
  'quantum_channel',
  'reset',
  'roerror',
  'save_amplitudes',
  'save_amplitudes_sq',
  'save_density_matrix',
  'save_expval',
  'save_expval_var',
  'save_probabilities',
  'save_probabilities_dict',
  'save_state',
  'save_statevector',
  'save_statevector_dict',
  'set_statevector',
  'switch_case',
  'while_loop'],
 'gates': [],
 'local': True,
 'simulator': True,
 'conditional': True,
 'memory': True,
 'max_shots': 1000000,
 'coupling_map': <qiskit.transpiler.coupling.CouplingMap at 0x7f0511fbd850>,
 'dynamic_reprate_enabled': False,
 'description': 'A C++ statevector simulator with noise',
 'url': 'https://github.com/Qiskit/qiskit-aer',
 'custom_instructions': ['break_loop',
  'continue_loop',
  'delay',
  '

In [ ]:
print(len(used_physical_qubits))
len(set([y for x in list(new_ess._coupling_map) for y in x]))

In [ ]:
len(backend.coupling_map.largest_connected_component())

In [ ]:
default_tcost_qc = transpile(rem, optimization_level=0, backend=backend)

In [ ]:
default_tcost_qc.num_qubits

In [ ]:
default_tcost_qc.draw(fold=-1, idle_wires=True)

In [19]:
filename = 'test_N3_W4_extra0_four0.0_six1.0.cvar0.05.p4.shots1000.initramp.d9'
with open(f'/lustre/scratch127/qpg/jc59/hubo/simulation.optimisation_{filename}.pkl', 'rb') as f:
    data = pickle.load(f)

In [20]:
history = data["history"]
hamiltonian: SparsePauliOp = data["hamiltonian"]
old_hamiltonian: SparsePauliOp = data["old_hamiltonian"]

In [21]:
results_file = '/lustre/scratch127/qpg/jc59/hubo/simulation_results_test_N3_W4_extra0_four0.0_six1.0.pkl'
with open(results_file, 'rb') as f:
    data = pickle.load(f)
    old_hamiltonian: SparsePauliOp = data['old_hamiltonian']
    hamiltonian: SparsePauliOp = data[9]['hamiltonian']
    edge_map = data[9]['layout']

In [22]:
map_old_hamiltonian = old_hamiltonian.apply_layout([edge_map[i] for i in range(old_hamiltonian.num_qubits)])

In [23]:
edge_map

{0: 10, 1: 8, 2: 6, 3: 0, 4: 2, 5: 4, 6: 11, 7: 9, 8: 7, 9: 1, 10: 3, 11: 5}

In [24]:
unmapping_edge_map = {val: key for key, val in edge_map.items()}
unmapping_edge_map

{10: 0, 8: 1, 6: 2, 0: 3, 2: 4, 4: 5, 11: 6, 9: 7, 7: 8, 1: 9, 3: 10, 5: 11}

Sample          Old_hamiltonian

                    | ~ Edge map, counting r2l 

                    v ~ pauli -> new_pauli, new_pauli[i] = pauli[unmapping_edge_map[11-i]]
                    
Mapped_sample   Hamiltonian

k = unmapping_edge_map[i]
=> edge_map[k] = i

sample -> mapped_sample, mapped_sample[i] = sample[11- unmapping_edge_map[11-i]]

In [25]:
print(old_hamiltonian.paulis[:4])
print(map_old_hamiltonian.paulis[:4])
# Apply layout counts r2l

['IIIIIIIIIIII', 'IIIIIZIIIIII', 'IIIIZIIIIIII', 'IIZIIIIIIIII']
['IIIIIIIIIIII', 'ZIIIIIIIIIII', 'IIZIIIIIIIII', 'IIIIIIIIIIZI']


In [26]:
for i in range(4):
    pauli = old_hamiltonian.paulis[i]
    print(pauli)
    print(''.join([str(pauli[unmapping_edge_map[11-j]]) for j in range(12)]))
    print(hamiltonian.paulis[i])
    print()

IIIIIIIIIIII
IIIIIIIIIIII
IIIIIIIIIIII

IIIIIZIIIIII
ZIIIIIIIIIII
ZIIIIIIIIIII

IIIIZIIIIIII
IIZIIIIIIIII
IIZIIIIIIIII

IIZIIIIIIIII
IIIIIIIIIIZI
IIIIIIIIIIZI



In [27]:
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [28]:
mapped_samples = history[-1][3]
unmapped_samples = {}
for sample, val in mapped_samples.items():
    unmapped_sample = ['0'] * 12
    for i in range(12):
        unmapped_sample[11- unmapping_edge_map[11-i]] = sample[i]
    unmapped_samples[''.join(unmapped_sample)] = val

In [29]:
sample = list(mapped_samples.keys())[0]
print(sample)
print(''.join([sample[edge_map[i]] for i in range(11,-1,-1)]))
''.join([sample[edge_map[i]] for i in range(11,-1,-1)])[::-1]

111011001111
101011111011


'110111110101'

In [30]:
evaluate_sparse_pauli_samples(
    list(mapped_samples.keys())[:20],
    hamiltonian
)

array([35., 38., 36., 23., 32., 22., 30., 31., 38., 39., 32., 36., 36.,
       21., 12., 39., 38., 33., 38., 32.])

In [31]:
evaluate_sparse_pauli_samples(
    list(mapped_samples.keys())[:20],
    map_old_hamiltonian
)

array([35., 38., 36., 23., 32., 22., 30., 31., 38., 39., 32., 36., 36.,
       21., 12., 39., 38., 33., 38., 32.])

In [32]:
sample_vals = evaluate_sparse_pauli_samples(
    list(unmapped_samples.keys()),
    old_hamiltonian
)
print(np.argmin(sample_vals))
print(np.min(sample_vals))
print(list(unmapped_samples.keys())[np.argmin(sample_vals)])

print(np.array(list(unmapped_samples.keys()))[np.nonzero([sample_vals <= 15])[1]])
print(sample_vals[np.nonzero([sample_vals <= 15])[1]])

# '100001100001' => 1, 4, 1, 4 => u0-, u2+, u0-, u2+
# '101000110100' => 5, 0, 3, 1 => u2-, u0+, u1-, u0-

220
0.0
000010101000
['000010101110' '101000010001' '001110100100' '110100001001'
 '001000010101' '101101000010' '000010101000' '101100001110'
 '100001110100' '000000010101' '001110100110' '000010110100'
 '101000101000' '000010000010' '010101101000' '100001110110'
 '000100001110' '000010010101']
[12. 12. 10. 12. 12. 12.  0. 12.  0. 10. 12. 12. 12. 12. 12. 12. 10. 12.]


In [ ]:
evaluate_sparse_pauli_samples(
    ['000010101000', '100 001 110 100'], 
    old_hamiltonian
)

array([0., 0.])

In [34]:
evaluate_sparse_pauli_samples(
    ['001000010001', '000010101000'],
    hamiltonian
)

array([ 0., 22.])

In [29]:
def indices_to_pauli(t, k, n, T):
    p = ['I'] * n * T
    p[t*n + k] = 'Z'
    return SparsePauliOp(''.join(p), 1)

In [44]:
indices_to_pauli(0, 2, 3, 4)

SparsePauliOp(['IIZIIIIIIIII'],
              coeffs=[1.+0.j])

In [43]:
old_hamiltonian[0:5]

SparsePauliOp(['IIIIIIIIIIII', 'ZIIIIIIIIIII', 'IIZIIIIIIIII', 'IIIIIIIIIIZI', 'ZIIIIIIIIIZI'],
              coeffs=[30.4375+0.j, -1.    +0.j, -1.    +0.j, -0.6875+0.j,  0.6875+0.j])